## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model_groq = init_chat_model("groq:openai/gpt-oss-20b")

In [2]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(description = "The Title of the movie")
    year: int = Field(description = "The year the movie was released")
    director: str = Field(description = "The director of the movie")
    rating: float = Field(description = "The rating of the movie out of 10")
    

In [4]:
model_with_structured_output = model_groq.with_structured_output(Movie)
model_with_structured_output

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021D0F8C62A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021D0F703CB0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The Titl

In [5]:
model_groq.invoke("Give info about Intersteller")

AIMessage(content='**Interstellar (2014)**  \n\n| Category | Details |\n|----------|---------|\n| **Director** | Christopher\u202fNolan |\n| **Screenplay** | Christopher\u202fNolan (adaptation of original story by Jonathan Nolan) |\n| **Producer(s)** | Christopher\u202fNolan, Emma Thomas, Lynda Obst |\n| **Music** | Hans\u202fZimmer |\n| **Cinematography** | Hoyte van Hoytema |\n| **Studio(s)** | Syncopy, Legendary Pictures, Warner Bros. Pictures |\n| **Distributor** | Warner Bros. Pictures |\n| **Release Dates** | 1\u202fNov\u202f2014 (US), 18\u202fNov\u202f2014 (UK) |\n| **Runtime** | 2\u202fh\u202f49\u202fmin |\n| **Budget** | ~\u202f$165\u202fmillion |\n| **Box‑Office** | $677\u202fmillion worldwide |\n| **Languages** | English |\n| **Genre** | Science‑fiction, Adventure, Drama |\n\n---\n\n### Synopsis (≈ 200\u202fwords)\n\nIn a dystopian near‑future Earth, a global crop‑failure crisis forces humanity to confront extinction. Former NASA pilot **Joseph Cooper** (Matthew\u202fMcConau

In [8]:
res = model_with_structured_output.invoke("Give info about Avengers Infinity War")
res

Movie(title='Avengers: Infinity War', year=2018, director='Anthony Russo, Joe Russo', rating=8.4)

## Message output alongside parsed structure

In [10]:
class Movie(BaseModel):
    """A Movie Details."""
    title: str = Field(description = "The Title of the movie")
    year: int = Field(description = "The year the movie was released")
    director: str = Field(description = "The director of the movie")
    rating: float = Field(description = "The rating of the movie out of 10")

model_with_structured_output = model_groq.with_structured_output(Movie, include_raw = True)
res = model_with_structured_output.invoke("Give info about Avengers Infinity War")
res


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "We need to provide info about Avengers: Infinity War. Use the function? There's a function Movie that takes director, rating, title, year. We can call that. Probably we should provide details: title, director, rating, year. Let's call function.", 'tool_calls': [{'id': 'fc_57a63fb7-2c7a-4891-a4f7-6da24c483236', 'function': {'arguments': '{"director":"Anthony Russo, Joe Russo","rating":8.5,"title":"Avengers: Infinity War","year":2018}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 163, 'total_tokens': 261, 'completion_time': 0.100462847, 'completion_tokens_details': {'reasoning_tokens': 53}, 'prompt_time': 0.007931602, 'prompt_tokens_details': None, 'queue_time': 0.338132204, 'total_time': 0.108394449}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_587d0d67cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs'

In [ ]:
### Nested Structured Output
class Actor(BaseModel):
    """An Actor Details."""
    name: str = Field(description = "The name of the actor")
    role: str = Field(description = "The role of the actor in the movie")
class MovieWithActors(BaseModel):
    """A Movie Details with Actors."""
    title: str = Field(description = "The Title of the movie")
    year: int = Field(description = "The year the movie was released")
    actors: list[Actor] = Field(description = "The list of actors in the movie")
    genre: list[str] = Field(description = "The genre of the movie")
    budget:float | None = Field(description = "budget in millions USD")

In [20]:
model_with_structured_output = model_groq.with_structured_output(MovieWithActors)
res = model_with_structured_output.invoke("Give info about captain america civil war")
res

MovieWithActors(title='Captain America: Civil War', year=2016, actors=[Actor(name='Chris Evans', role='Captain America / Steve Rogers'), Actor(name='Robert Downey Jr.', role='Iron Man / Tony Stark'), Actor(name='Scarlett Johansson', role='Black Widow / Natasha Romanoff'), Actor(name='Jeremy Renner', role='Hawkeye / Clint Barton'), Actor(name='Tom Holland', role='Spider-Man / Peter Parker'), Actor(name='Sebastian Stan', role='Bucky Barnes / Winter Soldier'), Actor(name='Anthony Mackie', role='Falcon / Sam Wilson'), Actor(name='Don Cheadle', role='War Machine / James Rhodes'), Actor(name='Paul Bettany', role='Vision'), Actor(name='Elizabeth Olsen', role='Loki'), Actor(name='Brie Larson', role='Carol Danvers / Captain Marvel')], genre=['Action', 'Adventure', 'Superhero'], budget=250000000.0)

## TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation

In [21]:
from typing_extensions import TypedDict,Annotated
class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]
model_withtypedict=model_groq.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [22]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model_groq.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie zack snyder's justice league")
response

{'budget': 300000000,
 'cast': [{'name': 'Henry Cavill', 'role': 'Superman'},
  {'name': 'Ben Affleck', 'role': 'Batman'},
  {'name': 'Ezra Miller', 'role': 'The Flash'},
  {'name': 'Jason Momoa', 'role': 'Aquaman'},
  {'name': 'Gal Gadot', 'role': 'Wonder Woman'},
  {'name': 'Ray Fisher', 'role': 'Cyborg'},
  {'name': 'Jesse Eisenberg', 'role': 'Lex Luthor'},
  {'name': 'Jeremy Irons', 'role': 'Zod'},
  {'name': 'Ciarán Hinds', 'role': 'Brainiac'},
  {'name': 'Zoe Saldana', 'role': 'Mera'}],
 'genres': ['Superhero', 'Action', 'Adventure', 'Science Fiction'],
 'title': "Zack Snyder's Justice League",
 'year': 2021}

In [23]:
model_groq.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [24]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
)
class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format = ContactInfo
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='308626b6-7ef4-4152-b991-f9b910b3c1e8'),
  AIMessage(content=[{'type': 'text', 'text': '{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', 'extras': {'signature': 'Et4KCtsKARFNMg9gZEm//+XcTyCSbZehMOxUYBAjqv++a473EUN57Pv3vLH04E1zgHbe4Zz37v8nRBlw0np825LlNk+A5D5TK7YmDl4nInQfv6pf/0SLmKP5q2ogSLz8ErRs1oQhM8SlpKn6Mxc/v/a3noFNPycroOtbmLMxwYaVdgIrumKsezly+ZZVxuHWzu0TbAdkEQ/ab66eUG6uedFrHH64WkrFu6Rx+S+VQS+ogpSrmEw0Y4GkVmSFuJalopXx0Q5tBQx2RCZCt7XlM7hssOQB9fvVRaHcB9n/ZysgxoF8Hnhvarkk6cDbSoU1rHL7rZscczytQUCzGwAPFcEOgt9NbRnF9FTHf3/xwaVUkSt3CoLSmgrXfTOvOfar9LRH4ZI+2Y9zDe78jUQ6Jp+zvGVz/fJ0NGglqfUVbJtRL9bykRnMqCLzhp3sXReV96Ek0WQA0WlhgMq3v3dcEJ2WOcEXnDhwWKJ4gzjlzb0rNh0hWtOuOgzBo/QP0gSuKeOaBwsleqSfPR/viJLNu2C96bNDEEfwdd6CBnBmYT4k/2hJ4aiGKqHnB9c8VbmcdZCrey2FoegsecllgD+eiAuR+jUGwVFDHEoiHF0IPHG6oG7rTEEygh0DpLWl/T3EHGYxQysH0p76tgsQyF

In [26]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [27]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model,
    response_format = ContactInfo
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})


result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}